In [1]:
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from database import get_engine

print("All imports successful!")

All imports successful!


In [2]:
engine = get_engine()

df = pd.read_sql("""
    SELECT date, ticker, open, high, low, close, volume
    FROM stock_prices
    ORDER BY ticker, date
""", engine)

df['date'] = pd.to_datetime(df['date'])

print(f"Total rows loaded: {len(df)}")
print(f"Tickers: {df['ticker'].unique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head(10)

Total rows loaded: 41505
Tickers: <StringArray>
[ 'AAPL',  'AMZN',   'BAC', 'GOOGL',    'GS',   'JNJ',   'JPM',    'MA',
  'META',  'MSFT',  'NVDA',   'SPY',   'UNH',     'V',   'XOM']
Length: 15, dtype: str
Date range: 2014-01-02 00:00:00 to 2024-12-30 00:00:00


,date,ticker,open,high,low,close,volume
0,2014-01-02,AAPL,17.203846,17.245641,17.090532,17.124897,234684800
1,2014-01-03,AAPL,17.116521,17.142526,16.731688,16.748716,392467600
2,2014-01-06,AAPL,16.639431,16.928906,16.520235,16.840052,412610800
3,2014-01-07,AAPL,16.852126,16.902900,16.653983,16.719618,317209200
4,2014-01-08,AAPL,16.681539,16.890519,16.677826,16.825504,258529600
5,2014-01-09,AAPL,16.928911,16.930769,16.574420,16.610643,279148800
6,2014-01-10,AAPL,16.713111,16.743143,16.443141,16.499798,304976000
7,2014-01-13,AAPL,16.405989,16.795774,16.405060,16.586176,378492800
8,2014-01-14,AAPL,16.663274,16.926744,16.645936,16.916218,332561600
9,2014-01-15,AAPL,17.136958,17.343771,17.079372,17.255844,391638800


In [ ]:
def calculate_momentum_factors(df):
    df = df.sort_values(['ticker', 'date']).copy()
    
    grouped = df.groupby('ticker')['close']
    
    df['mom_12m'] = grouped.transform(lambda x: x.pct_change(252))
    df['mom_6m']  = grouped.transform(lambda x: x.pct_change(126))
    df['mom_1m']  = grouped.transform(lambda x: x.pct_change(21))
    df['mom_5d']  = grouped.transform(lambda x: x.pct_change(5))
    
    df['reversal'] = -df['mom_5d']
    
    return df

df = calculate_momentum_factors(df)
print("Momentum factors calculated!")
df[['date', 'ticker', 'close', 'mom_12m', 'mom_6m', 'mom_1m', 'reversal']].tail(10)

Momentum factors calculated!


,date,ticker,close,mom_12m,mom_6m,mom_1m,reversal
41495,2024-12-16,XOM,103.236450,0.104253,0.017366,-0.100282,0.039238
41496,2024-12-17,XOM,102.798653,0.106981,0.003605,-0.094711,0.041360
41497,2024-12-18,XOM,101.285355,0.082638,-0.032054,-0.115452,0.049142
41498,2024-12-19,XOM,100.419266,0.059415,-0.031840,-0.110596,0.056430
41499,2024-12-20,XOM,100.761909,0.081084,-0.056560,-0.120096,0.044839
41500,2024-12-23,XOM,101.171150,0.080567,-0.055379,-0.128188,0.020006
41501,2024-12-24,XOM,101.266319,0.079673,-0.054821,-0.126365,0.014906
41502,2024-12-26,XOM,101.351982,0.078153,-0.058055,-0.112361,-0.000658
41503,2024-12-27,XOM,101.342461,0.083142,-0.059944,-0.097398,-0.009193
41504,2024-12-30,XOM,100.657204,0.091602,-0.065001,-0.101139,0.001039


In [5]:
def calculate_volatility_factors(df):
    df = df.sort_values(['ticker', 'date']).copy()
    
    grouped = df.groupby('ticker')['close']
    
    # Daily returns
    df['daily_return'] = grouped.transform(lambda x: x.pct_change())
    
    # Realized volatility - std of daily returns over 21 days, annualized
    df['vol_21d'] = df.groupby('ticker')['daily_return'].transform(
        lambda x: x.rolling(21).std() * np.sqrt(252)
    )
    
    # Long term volatility
    df['vol_63d'] = df.groupby('ticker')['daily_return'].transform(
        lambda x: x.rolling(63).std() * np.sqrt(252)
    )
    
    # Volatility ratio - is the stock more volatile than usual?
    df['vol_ratio'] = df['vol_21d'] / df['vol_63d']
    
    return df

df = calculate_volatility_factors(df)
print("Volatility factors calculated!")
df[['date', 'ticker', 'close', 'daily_return', 'vol_21d', 'vol_63d', 'vol_ratio']].tail(10)

Volatility factors calculated!


,date,ticker,close,daily_return,vol_21d,vol_63d,vol_ratio
41495,2024-12-16,XOM,103.236450,-0.021382,0.171520,0.199282,0.860691
41496,2024-12-17,XOM,102.798653,-0.004241,0.170397,0.199235,0.855259
41497,2024-12-18,XOM,101.285355,-0.014721,0.166855,0.199356,0.836970
41498,2024-12-19,XOM,100.419266,-0.008551,0.164544,0.199643,0.824191
41499,2024-12-20,XOM,100.761909,0.003412,0.151949,0.195989,0.775296
41500,2024-12-23,XOM,101.171150,0.004061,0.139900,0.196278,0.712764
41501,2024-12-24,XOM,101.266319,0.000941,0.141084,0.192819,0.731691
41502,2024-12-26,XOM,101.351982,0.000846,0.139595,0.190026,0.734608
41503,2024-12-27,XOM,101.342461,-0.000094,0.134792,0.181644,0.742069
41504,2024-12-30,XOM,100.657204,-0.006762,0.134701,0.179969,0.748466


In [6]:
def calculate_volume_factors(df):
    df = df.sort_values(['ticker', 'date']).copy()
    
    # Average volume over 21 days
    df['vol_avg_21d'] = df.groupby('ticker')['volume'].transform(
        lambda x: x.rolling(21).mean()
    )
    
    # Volume ratio - is today's volume unusual?
    df['volume_ratio'] = df['volume'] / df['vol_avg_21d']
    
    # Price volume trend - combines price direction and volume
    df['daily_return'] = df.groupby('ticker')['close'].transform(
        lambda x: x.pct_change()
    )
    df['pvt'] = df.groupby('ticker').apply(
        lambda x: (x['daily_return'] * x['volume']).cumsum()
    ).reset_index(level=0, drop=True)
    
    return df

df = calculate_volume_factors(df)
print("Volume factors calculated!")
df[['date', 'ticker', 'volume', 'vol_avg_21d', 'volume_ratio', 'pvt']].tail(10)

Volume factors calculated!


,date,ticker,volume,vol_avg_21d,volume_ratio,pvt
41495,2024-12-16,XOM,20256100,1.622556e+07,1.248407,1.625058e+07
41496,2024-12-17,XOM,17554000,1.615424e+07,1.086650,1.617613e+07
41497,2024-12-18,XOM,17114500,1.629113e+07,1.050541,1.592419e+07
41498,2024-12-19,XOM,20565600,1.671881e+07,1.230088,1.574834e+07
41499,2024-12-20,XOM,40141200,1.808930e+07,2.219058,1.588530e+07
41500,2024-12-23,XOM,12285100,1.797548e+07,0.683437,1.593520e+07
41501,2024-12-24,XOM,7807000,1.771279e+07,0.440755,1.594254e+07
41502,2024-12-26,XOM,9652400,1.690670e+07,0.570922,1.595071e+07
41503,2024-12-27,XOM,11943900,1.676940e+07,0.712244,1.594958e+07
41504,2024-12-30,XOM,11080800,1.676948e+07,0.660772,1.587466e+07


In [7]:
def calculate_mean_reversion_factors(df):
    df = df.sort_values(['ticker', 'date']).copy()
    
    # Distance from 52 week high
    df['high_52w'] = df.groupby('ticker')['close'].transform(
        lambda x: x.rolling(252).max()
    )
    df['dist_from_high'] = (df['close'] - df['high_52w']) / df['high_52w']
    
    # RSI - Relative Strength Index
    def compute_rsi(series, period=14):
        delta = series.diff()
        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)
        avg_gain = gain.rolling(period).mean()
        avg_loss = loss.rolling(period).mean()
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
        return rsi
    
    df['rsi'] = df.groupby('ticker')['close'].transform(compute_rsi)
    
    return df

df = calculate_mean_reversion_factors(df)
print("Mean reversion factors calculated!")
df[['date', 'ticker', 'close', 'high_52w', 'dist_from_high', 'rsi']].tail(10)

Mean reversion factors calculated!


,date,ticker,close,high_52w,dist_from_high,rsi
41495,2024-12-16,XOM,103.236450,118.348557,-0.127692,6.106745
41496,2024-12-17,XOM,102.798653,118.348557,-0.131391,6.920282
41497,2024-12-18,XOM,101.285355,118.348557,-0.144178,6.230393
41498,2024-12-19,XOM,100.419266,118.348557,-0.151496,3.717533
41499,2024-12-20,XOM,100.761909,118.348557,-0.148600,6.277519
41500,2024-12-23,XOM,101.171150,118.348557,-0.145143,9.247361
41501,2024-12-24,XOM,101.266319,118.348557,-0.144338,13.039427
41502,2024-12-26,XOM,101.351982,118.348557,-0.143615,9.560980
41503,2024-12-27,XOM,101.342461,118.348557,-0.143695,10.828730
41504,2024-12-30,XOM,100.657204,118.348557,-0.149485,10.769235


In [8]:
# Combine all factors into one DataFrame
factor_cols = [
    'date', 'ticker', 'close', 'daily_return',
    'mom_12m', 'mom_6m', 'mom_1m', 'mom_5d', 'reversal',
    'vol_21d', 'vol_63d', 'vol_ratio',
    'volume_ratio', 'pvt',
    'high_52w', 'dist_from_high', 'rsi'
]

factors_df = df[factor_cols].copy()
factors_df = factors_df.dropna()

print(f"Total rows with complete factors: {len(factors_df)}")
print(f"Date range: {factors_df['date'].min()} to {factors_df['date'].max()}")
factors_df.tail(5)

Total rows with complete factors: 37725
Date range: 2015-01-02 00:00:00 to 2024-12-30 00:00:00


,date,ticker,close,daily_return,mom_12m,mom_6m,mom_1m,mom_5d,reversal,vol_21d,vol_63d,vol_ratio,volume_ratio,pvt,high_52w,dist_from_high,rsi
41500,2024-12-23,XOM,101.171150,0.004061,0.080567,-0.055379,-0.128188,-0.020006,0.020006,0.139900,0.196278,0.712764,0.683437,1.593520e+07,118.348557,-0.145143,9.247361
41501,2024-12-24,XOM,101.266319,0.000941,0.079673,-0.054821,-0.126365,-0.014906,0.014906,0.141084,0.192819,0.731691,0.440755,1.594254e+07,118.348557,-0.144338,13.039427
41502,2024-12-26,XOM,101.351982,0.000846,0.078153,-0.058055,-0.112361,0.000658,-0.000658,0.139595,0.190026,0.734608,0.570922,1.595071e+07,118.348557,-0.143615,9.560980
41503,2024-12-27,XOM,101.342461,-0.000094,0.083142,-0.059944,-0.097398,0.009193,-0.009193,0.134792,0.181644,0.742069,0.712244,1.594958e+07,118.348557,-0.143695,10.828730
41504,2024-12-30,XOM,100.657204,-0.006762,0.091602,-0.065001,-0.101139,-0.001039,0.001039,0.134701,0.179969,0.748466,0.660772,1.587466e+07,118.348557,-0.149485,10.769235


In [9]:
FACTOR_COLS = [
    'date', 'ticker', 'close', 'daily_return',
    'mom_12m', 'mom_6m', 'mom_1m', 'mom_5d', 'reversal',
    'vol_21d', 'vol_63d', 'vol_ratio',
    'volume_ratio', 'pvt',
    'high_52w', 'dist_from_high', 'rsi'
]

factors_df = df[FACTOR_COLS].copy()
factors_df = factors_df.dropna()
factors_df['date'] = pd.to_datetime(factors_df['date']).dt.date

print(f"Rows to save: {len(factors_df):,}")
print(f"Date range: {factors_df['date'].min()} to {factors_df['date'].max()}")

factors_df.to_sql(
    'factors',
    engine,
    if_exists='replace',
    index=False,
    method='multi'
)

print("All factors saved to database!")

Rows to save: 37,725
Date range: 2015-01-02 to 2024-12-30
All factors saved to database!
